In [19]:
# First, I'll import all necessary libraries for my Spark ML implementation
import pyspark
from pyspark import SparkContext
from pyspark.sql import SQLContext
import os
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np


In [21]:

# Setting up my Spark environment - this creates the basic execution context
# that will handle distributed processing
# Check if there is an existing SparkContext
if 'sc' not in globals():
    sc = SparkContext()
    sqlContext = SQLContext(sc)
else:
    sqlContext = SQLContext(sc)

# I'm loading the Iris dataset from the provided path
# This dataset contains measurements of different iris species
data_path = ""
iris_file = os.path.join(data_path, "iris.csv")

# Reading the CSV file using pandas first, then converting to Spark DataFrame
# This two-step approach gives me more flexibility with initial data handling
iris_pandas = pd.read_csv(iris_file)
iris_df = sqlContext.createDataFrame(iris_pandas)


/home/coolmax/anaconda3/envs/pyspark_env/lib/python3.8/site-packages/pyspark/sql/context.py:113: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


Py4JJavaError: An error occurred while calling o49.sessionState.
: java.lang.IllegalStateException: LiveListenerBus is stopped.
	at org.apache.spark.scheduler.LiveListenerBus.addToQueue(LiveListenerBus.scala:92)
	at org.apache.spark.scheduler.LiveListenerBus.addToStatusQueue(LiveListenerBus.scala:75)
	at org.apache.spark.sql.internal.SharedState.<init>(SharedState.scala:115)
	at org.apache.spark.sql.SparkSession.$anonfun$sharedState$1(SparkSession.scala:143)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.SparkSession.sharedState$lzycompute(SparkSession.scala:143)
	at org.apache.spark.sql.SparkSession.sharedState(SparkSession.scala:142)
	at org.apache.spark.sql.SparkSession.$anonfun$sessionState$2(SparkSession.scala:162)
	at scala.Option.getOrElse(Option.scala:189)
	at org.apache.spark.sql.SparkSession.sessionState$lzycompute(SparkSession.scala:160)
	at org.apache.spark.sql.SparkSession.sessionState(SparkSession.scala:157)
	at java.base/jdk.internal.reflect.Native

In [ ]:

# Let's check the schema to understand the data structure
print("Dataset schema:")
iris_df.printSchema()

# I need to convert the text labels to numeric values for machine learning
# StringIndexer handles this encoding automatically
from pyspark.ml.feature import StringIndexer

# Creating an indexer to transform "variety" column to numeric "label" column
label_indexer = StringIndexer(inputCol="variety", outputCol="label")
indexer_model = label_indexer.fit(iris_df)
iris_labeled = indexer_model.transform(iris_df)

# Let's see what numeric values were assigned to each species
print("Label mapping:")
iris_labeled.select("variety", "label").distinct().show()

# Now I'll create a feature vector combining all measurement columns
# This is required by Spark ML algorithms that expect a single features column
from pyspark.ml.feature import VectorAssembler

# Combining the four measurement features into a single vector column
feature_cols = ["sepal_length", "sepal_width", "petal_length", "petal_width"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
iris_features = assembler.transform(iris_labeled)

# Taking a quick look at our prepared dataset
print("First 5 records with features vector:")
iris_features.select("variety", "label", "features").show(5)

# Caching the dataset for faster processing since I'll use it multiple times
iris_features.cache()


In [ ]:

# Splitting data into training (90%) and testing (10%) sets
# The random split uses a seed for reproducibility
train_data, test_data = iris_features.randomSplit([0.9, 0.1], seed=42)
print(f"Training set size: {train_data.count()} rows")
print(f"Test set size: {test_data.count()} rows")

# Now I'll implement three different ML algorithms to compare performance
from pyspark.ml.classification import DecisionTreeClassifier, RandomForestClassifier, GBTClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# Setting up the evaluator for consistent measurement across models
evaluator = MulticlassClassificationEvaluator(
    predictionCol="prediction", 
    labelCol="label", 
    metricName="accuracy"
)

# 1. Decision Tree - A simple but intuitive model
print("\n--- Decision Tree Classifier ---")
dt_classifier = DecisionTreeClassifier(
    maxDepth=4,          # Preventing overfitting by limiting tree depth
    labelCol="label",    # Target variable
    featuresCol="features"  # Input features
)

# Training the decision tree model
dt_model = dt_classifier.fit(train_data)
print(f"Tree depth: {dt_model.depth}")
print(f"Number of nodes: {dt_model.numNodes}")

# Making predictions on test data
dt_predictions = dt_model.transform(test_data)

# Evaluating accuracy
dt_accuracy = evaluator.evaluate(dt_predictions)
print(f"Decision Tree accuracy: {dt_accuracy:.4f}")

# Generating confusion matrix to analyze prediction patterns
print("Confusion matrix:")
dt_predictions.groupBy("label", "prediction").count().show()

# 2. Random Forest - An ensemble of decision trees
print("\n--- Random Forest Classifier ---")
rf_classifier = RandomForestClassifier(
    numTrees=10,        # Using 10 trees in the ensemble
    labelCol="label", 
    featuresCol="features"
)

# Training the random forest model
rf_model = rf_classifier.fit(train_data)

# Making predictions and evaluating
rf_predictions = rf_model.transform(test_data)
rf_accuracy = evaluator.evaluate(rf_predictions)
print(f"Random Forest accuracy: {rf_accuracy:.4f}")

# Confusion matrix for random forest
print("Confusion matrix:")
rf_predictions.groupBy("label", "prediction").count().show()

# 3. Gradient Boosted Trees - A sequential ensemble approach
print("\n--- Gradient Boosted Trees Classifier ---")
gbt_classifier = GBTClassifier(
    maxIter=10,          # Number of boosting iterations
    labelCol="label", 
    featuresCol="features"
)

# Training the GBT model
gbt_model = gbt_classifier.fit(train_data)

# Making predictions and evaluating
gbt_predictions = gbt_model.transform(test_data)
gbt_accuracy = evaluator.evaluate(gbt_predictions)
print(f"Gradient Boosted Trees accuracy: {gbt_accuracy:.4f}")

# Confusion matrix for GBT
print("Confusion matrix:")
gbt_predictions.groupBy("label", "prediction").count().show()

# Comparing all model accuracies
print("\n--- Model Comparison ---")
models = ["Decision Tree", "Random Forest", "Gradient Boosted Trees"]
accuracies = [dt_accuracy, rf_accuracy, gbt_accuracy]

# Creating a simple bar chart to visualize the comparison
plt.figure(figsize=(10, 6))
plt.bar(models, accuracies, color=['skyblue', 'lightgreen', 'salmon'])
plt.title('Model Accuracy Comparison')
plt.ylabel('Accuracy')
plt.ylim([0, 1])  # Setting y-axis from 0 to 1 for proper perspective
plt.grid(axis='y', linestyle='--', alpha=0.7)
for i, acc in enumerate(accuracies):
    plt.text(i, acc+0.02, f'{acc:.4f}', ha='center')
plt.show()


In [11]:

# Finally, cleaning up Spark context
# This is important to release resources when the job is done
sc.stop()